# Assignment 5 — Solution

## Designing a RAG System for a Hospital

**Module 1 · Lesson 5 — Vector Databases & ANN Search**

**Author:** Naeem Naseer

---

### The Brief

Design a RAG system for a hospital with:

| Requirement | Value |
| --- | --- |
| Documents | **5 million** medical documents |
| Users | Doctors, searching in natural language |
| Metadata | Department, Country, Language, Publication Date |

And answer:

1. Why would you choose a vector database?
2. Which retrieval method would you use?
3. Would you use hybrid search? Why?
4. How would metadata filtering improve performance?
5. Why is ANN preferable to exact search here?
6. *(Harder)* Compute RAM for 5M vectors at 1024 dims. Does it fit on one machine?
7. *(Harder)* Cardiology is 2% of documents. What goes wrong with post-filtering?
8. *(Harder)* Give a query where hybrid search wins and pure vector search fails.

> **This notebook does not just answer in prose.** Sections 4–7 build a working search
> engine and *measure* the ANN speed/recall trade-off, the post-filter failure, and the
> hybrid-search win. Every number below was produced by running the code.

---

## 1. Architecture

### Flow A — Ingestion (once per document)

```text
 5,000,000 medical documents (PDF, DICOM reports, HL7, discharge notes)
                    │
                    ▼
        ┌───────────────────────┐
        │  Document Storage     │   S3 with encryption at rest
        │  (original files)     │   retained for citation + reprocessing
        └───────────┬───────────┘
                    ▼
             Background Workers          (Celery / ARQ — never in the request path)
                    │
                    ├──► PHI detection & handling      ◄── healthcare-specific
                    │      flag patient identifiers before they reach any index
                    │
                    ├──► Text extraction + cleaning
                    │
                    ├──► Structure-aware chunking
                    │      400–600 tokens, 15% overlap (dense clinical text)
                    │      prepend "Department > Section" to each chunk
                    │
                    ├──► Embedding model  (biomedical, e.g. BioBERT/PubMedBERT-based)
                    │      the SAME model + prefix convention used at query time
                    │
        ┌───────────┴───────────┐
        ▼                       ▼
┌────────────────┐    ┌──────────────────┐
│ Vector DB      │    │ BM25 Index       │   both are needed — see §7
│ (Qdrant, HNSW) │    │ (Elasticsearch)  │
│ vector+payload │    │ exact tokens     │
└────────────────┘    └──────────────────┘
```

### Flow B — Query (every doctor's question)

```text
        Doctor: "What is the first-line treatment for myocardial infarction?"
                    │
                    ▼
            FastAPI Backend
                    │
                    ├──► AuthN + AuthZ        ← which departments/countries may this
                    │                            doctor see? becomes a metadata filter
                    ├──► Input guardrails
                    │
                    ▼
            Query Embedding Model     ← same model + prefix as ingestion
                    │
        ┌───────────┴───────────┐
        ▼                       ▼
   Vector search           BM25 search
   + metadata PRE-filter    (exact codes, drug names)
   (filterable HNSW)
   top 50                   top 50
        └───────────┬───────────┘
                    ▼
          Reciprocal Rank Fusion
                    │
                    ▼
          Cross-encoder Reranker   (50 → 5)
                    │
                    ▼
            Prompt Builder
                    │
                    ▼
                   LLM
                    │
                    ▼
     Grounding check + citation validation    ← non-negotiable in clinical settings
                    │
                    ▼
       Answer + source documents + dates
```

> ⚕️ **Two healthcare-specific additions** that a generic RAG diagram would miss:
> **PHI handling at ingestion** (patient identifiers must be detected before indexing, and
> the vector DB counts as personal data — see the embedding-inversion note in Lesson 4),
> and **publication-date surfacing**, because a 2015 treatment protocol that has since
> been superseded is not merely unhelpful, it is dangerous. Date must be both a filter and
> a visible part of every citation.

---

## 2. Answers to the Core Questions

### Q1 — Why choose a vector database?

Doctors search in **natural language**, and clinical language is full of synonym pairs
where the two forms share no characters:

| Doctor types | Document says |
| --- | --- |
| heart attack | myocardial infarction |
| high blood pressure | hypertension |
| kidney failure | renal insufficiency |
| painkiller | analgesic |

SQL `LIKE` and exact matching find **none** of these. A vector database indexes embeddings
so retrieval works on *meaning*.

The second reason is scale. You could store vectors in a Postgres column and compute
cosine similarity in SQL — but that is a full scan of 5 million rows on every query.
A vector database adds an **ANN index** that avoids examining most of the data. Section 5
measures exactly how much that matters.

### Q2 — Which retrieval method?

**HNSW**, with:

- `M = 16`, `ef_construction = 200` at build time
- `ef_search` tuned at query time to hit ≥ 0.95 recall (measured, not assumed)
- **Filterable HNSW** so metadata filters apply *during* graph traversal (see Q7)
- Scalar quantisation (float32 → int8) with full-precision rescoring of the top 100

At 5M vectors HNSW is comfortably the right choice: IVF suits large static datasets but
handles updates poorly, and hospital documents are updated continuously as protocols change.

### Q3 — Would you use hybrid search?

**Yes — mandatory in this domain.** Medical retrieval is full of exact tokens that
embeddings blur together:

- Drug names: *"metoprolol"* vs *"metformin"* — one letter apart in effect, unrelated in meaning
- ICD-10 codes: `I21.9` (acute MI) vs `I21.0`
- Protocol identifiers: `PROTOCOL-CARD-2026-014`
- Dosages: *"5mg"* vs *"50mg"*

Getting a drug name approximately right is a patient-safety failure, not a relevance
failure. BM25 handles exact tokens precisely; embeddings handle the synonym problem from
Q1. Fuse with RRF. **Section 7 demonstrates this failing and being fixed.**

### Q4 — How does metadata filtering improve performance?

Three distinct benefits, and the third is the most important:

1. **Speed** — filtering `Department = Cardiology` cuts the candidate set from 5M to
   ~100k, roughly a 50× smaller search space.
2. **Accuracy** — a paediatric dosage retrieved for an adult cardiology question is
   actively harmful. `Language` and `Publication Date` filters keep results in-scope, and
   date filtering prevents superseded protocols from surfacing.
3. **Security and access control** — this is the real reason. A doctor's permissions
   become a mandatory metadata filter. This is the same mechanism that stopped the
   cross-tenant leak in the Lesson 1 assignment, and under HIPAA/GDPR it is a legal
   requirement rather than an optimisation.

### Q5 — Why ANN over exact search?

Exact search compares the query against all 5 million vectors on **every** query. Section 5
measures this directly. The trade — a few percent of recall for a large speed gain — is
easy to accept, because losing the 10th-best chunk rarely changes the answer while a
multi-second wait definitely changes the doctor's experience.

The critical discipline is that "a few percent" must be **measured**, not assumed. Section 6
measures it.

---

## 3. Q6 — Capacity Planning

Before writing any code: does this fit on one machine?

In [1]:
def capacity(n_vectors, dims, bytes_per_value=4, hnsw_M=16, label=""):
    """RAM estimate for an HNSW index.

    Two components people forget:
      - the HNSW graph itself (~M links x 2 x 4 bytes per node, plus overhead)
      - the payload (original text + metadata), which must live somewhere
    """
    vectors_gb = n_vectors * dims * bytes_per_value / 1024**3
    graph_gb   = n_vectors * hnsw_M * 2 * 4 / 1024**3        # bidirectional links
    payload_gb = n_vectors * 1200 / 1024**3                  # ~1.2 KB text+metadata/chunk

    print(label)
    print("  vectors        : %8.2f GB   (%d x %d x %d bytes)"
          % (vectors_gb, n_vectors, dims, bytes_per_value))
    print("  HNSW graph     : %8.2f GB" % graph_gb)
    print("  payload/text   : %8.2f GB" % payload_gb)
    print("  " + "-" * 46)
    print("  TOTAL          : %8.2f GB" % (vectors_gb + graph_gb + payload_gb))
    print()
    return vectors_gb + graph_gb + payload_gb


# The brief says 5 million DOCUMENTS. Documents are chunked before embedding,
# and this is the assumption most people forget to state.
CHUNKS_PER_DOC = 8
n_docs = 5_000_000
n_chunks = n_docs * CHUNKS_PER_DOC

print("5,000,000 documents x %d chunks/doc = %s chunks to index" % (CHUNKS_PER_DOC, "{:,}".format(n_chunks)))
print("=" * 58)
print()

capacity(n_chunks, 1024, 4, label="A) 1024 dims, float32 (as the question asks)")
capacity(n_chunks, 1024, 1, label="B) 1024 dims, int8 scalar quantisation (4x smaller)")
capacity(n_chunks, 384,  4, label="C) 384 dims, float32 (smaller model)")

5,000,000 documents x 8 chunks/doc = 40,000,000 chunks to index

A) 1024 dims, float32 (as the question asks)
  vectors        :   152.59 GB   (40000000 x 1024 x 4 bytes)
  HNSW graph     :     4.77 GB
  payload/text   :    44.70 GB
  ----------------------------------------------
  TOTAL          :   202.06 GB

B) 1024 dims, int8 scalar quantisation (4x smaller)
  vectors        :    38.15 GB   (40000000 x 1024 x 1 bytes)
  HNSW graph     :     4.77 GB
  payload/text   :    44.70 GB
  ----------------------------------------------
  TOTAL          :    87.62 GB

C) 384 dims, float32 (smaller model)
  vectors        :    57.22 GB   (40000000 x 384 x 4 bytes)
  HNSW graph     :     4.77 GB
  payload/text   :    44.70 GB
  ----------------------------------------------
  TOTAL          :   106.69 GB



### Answer to Q6

**The question hides a trap, and spotting it is the point.** "5 million documents" is not
5 million vectors — documents are chunked first. At ~8 chunks per document that is
**40 million vectors**, and the honest answer starts by stating that assumption.

At 1024 dims float32 the index needs roughly **200 GB**. That does *not* fit a typical
server, but it is not exotic either — it fits comfortably on a single large-memory cloud
instance (AWS `r6i.8xlarge` = 256 GB, or `r6i.12xlarge` = 384 GB with headroom).

**Three ways to bring it down, in order of preference:**

1. **Scalar quantisation** (float32 → int8) — 4× smaller vectors, minimal recall loss,
   with full-precision rescoring of the top 100. This alone gets it near 90 GB.
2. **A smaller embedding model** — 384 dims instead of 1024 is another large saving, *if*
   retrieval quality holds on a medical golden set. Measure before deciding.
3. **Sharding** across nodes by `Department` or `Country` — which is attractive here
   anyway, because those are exactly the fields doctors filter on.

**What I would actually do:** shard by Country (data-residency law often requires it for
medical records regardless), and apply scalar quantisation within each shard. That turns a
200 GB single-node problem into several comfortable nodes with a compliance benefit.

---

## 4. Building a Working Search Engine

Everything from here on is measured, not asserted. We build a corpus, index it three
different ways, and benchmark.

Real 40M-vector benchmarks need a cluster, so we scale down to 100,000 vectors — the
*relationships* (ANN vs exact, recall vs speed, filter behaviour) hold at any scale.

In [2]:
import numpy as np
import time

rng = np.random.default_rng(7)

N_VECTORS = 1_000_000
DIMS = 128


print("Building a synthetic corpus of %s vectors x %d dims..." % ("{:,}".format(N_VECTORS), DIMS))

# Real embeddings are CLUSTERED by topic, and this matters enormously for the
# benchmark: uniformly random vectors are the WORST possible case for any ANN index,
# and measuring against them makes ANN look far worse than it is in practice.
# So we build tight topic clusters (noise norm ~0.57 vs a unit-length centre).
n_topics = 200
topic_centres = rng.normal(size=(n_topics, DIMS)).astype(np.float32)
topic_centres /= np.linalg.norm(topic_centres, axis=1, keepdims=True)

assignments = rng.integers(0, n_topics, size=N_VECTORS)
vectors = topic_centres[assignments] + rng.normal(scale=0.05, size=(N_VECTORS, DIMS)).astype(np.float32)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)     # normalise (Lesson 4 §9)

# Metadata. Cardiology is deliberately rare (2%) - that is the Q7 scenario.
departments = np.empty(N_VECTORS, dtype=object)
dept_names = ["Radiology", "Oncology", "Pediatrics", "Neurology", "Surgery",
              "Emergency", "Pharmacy", "Pathology", "Orthopedics", "Psychiatry",
              "Billing", "Cardiology"]
probs = np.array([0.14, 0.12, 0.10, 0.09, 0.09, 0.09, 0.08, 0.08, 0.07, 0.07, 0.05, 0.02])
dept_idx = rng.choice(len(dept_names), size=N_VECTORS, p=probs)
departments = np.array(dept_names, dtype=object)[dept_idx]

print("Done.")
print()
print("Department distribution:")
for i, name in enumerate(dept_names):
    count = int((dept_idx == i).sum())
    print("  %-13s %7s  (%4.1f%%)" % (name, "{:,}".format(count), 100 * count / N_VECTORS))

cardiology_count = int((departments == "Cardiology").sum())
print()
print("Cardiology is %.1f%% of the corpus - deliberately rare, for the Q7 demonstration."
      % (100 * cardiology_count / N_VECTORS))

Building a synthetic corpus of 1,000,000 vectors x 128 dims...
Done.

Department distribution:
  Radiology     140,201  (14.0%)
  Oncology      119,953  (12.0%)
  Pediatrics     99,915  (10.0%)
  Neurology      89,556  ( 9.0%)
  Surgery        89,880  ( 9.0%)
  Emergency      90,341  ( 9.0%)
  Pharmacy       80,215  ( 8.0%)
  Pathology      79,750  ( 8.0%)
  Orthopedics    69,790  ( 7.0%)
  Psychiatry     70,370  ( 7.0%)
  Billing        50,183  ( 5.0%)
  Cardiology     19,846  ( 2.0%)

Cardiology is 2.0% of the corpus - deliberately rare, for the Q7 demonstration.


---

## 5. Q5 — Exact Search vs ANN, Measured

First the baseline: **exact (brute-force) search**. Because our vectors are normalised,
cosine similarity is a single matrix multiply (Lesson 4 §9).

In [3]:
def exact_search(query, vectors, k=10):
    """Brute force. 100% recall by definition - this is our ground truth."""
    scores = vectors @ query                       # normalised => dot product == cosine
    top = np.argpartition(-scores, k)[:k]
    return top[np.argsort(-scores[top])]


# Benchmark over a set of queries so one unlucky query cannot skew the result
N_QUERIES = 50
queries = topic_centres[rng.integers(0, n_topics, size=N_QUERIES)] + \
          rng.normal(scale=0.05, size=(N_QUERIES, DIMS)).astype(np.float32)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

start = time.perf_counter()
ground_truth = [exact_search(q, vectors, k=10) for q in queries]
exact_elapsed = time.perf_counter() - start

exact_ms = 1000 * exact_elapsed / N_QUERIES
print("EXACT SEARCH (brute force)")
print("  corpus size      : %s vectors" % "{:,}".format(N_VECTORS))
print("  queries run      : %d" % N_QUERIES)
print("  time per query   : %.2f ms" % exact_ms)
print("  recall@10        : 1.000  (it IS the ground truth)")
print()
print("Extrapolating to the real corpus of 40M chunks (40x larger):")
print("  estimated        : %.0f ms per query  (~%.1f seconds)"
      % (exact_ms * 40, exact_ms * 40 / 1000))
print()
print(">>> And that is a SINGLE query with no other load on the machine.")
print(">>> With concurrent doctors it collapses. This is why ANN exists.")

EXACT SEARCH (brute force)
  corpus size      : 1,000,000 vectors
  queries run      : 50
  time per query   : 19.54 ms
  recall@10        : 1.000  (it IS the ground truth)

Extrapolating to the real corpus of 40M chunks (40x larger):
  estimated        : 782 ms per query  (~0.8 seconds)

>>> And that is a SINGLE query with no other load on the machine.
>>> With concurrent doctors it collapses. This is why ANN exists.


### Now an ANN index (IVF)

We implement **IVF (Inverted File)**: cluster the vectors, then search only the clusters
nearest the query. It's simpler to write than HNSW but demonstrates the identical
principle — *look at a fraction of the data* — and has the same kind of tuning knob.

| | IVF | HNSW |
| --- | --- | --- |
| Structure | Clusters | Layered graph |
| Query knob | `nprobe` (clusters searched) | `ef_search` (candidates explored) |
| Trade-off | **Identical**: more search → better recall, slower query |

In [4]:
class IVFIndex:
    """Inverted File index: cluster the space, search only the nearest clusters."""

    def __init__(self, n_clusters=2048, seed=0):
        self.n_clusters = n_clusters
        self.rng = np.random.default_rng(seed)

    def build(self, vectors, iterations=8, sample=40_000):
        n = len(vectors)
        # k-means on a sample: full k-means on the whole corpus is unnecessary
        sample_idx = self.rng.choice(n, size=min(sample, n), replace=False)
        sample_vecs = vectors[sample_idx]
        centroids = sample_vecs[self.rng.choice(len(sample_vecs), self.n_clusters, replace=False)].copy()

        for _ in range(iterations):
            assign = np.argmax(sample_vecs @ centroids.T, axis=1)
            for c in range(self.n_clusters):
                members = sample_vecs[assign == c]
                if len(members):
                    centroids[c] = members.mean(axis=0)
            centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)

        self.centroids = centroids

        # Assign every vector to its nearest centroid, in chunks to bound memory
        assignments = np.empty(n, dtype=np.int32)
        for start in range(0, n, 10_000):
            block = vectors[start:start + 10_000]
            assignments[start:start + 10_000] = np.argmax(block @ centroids.T, axis=1)

        self.lists = [np.where(assignments == c)[0] for c in range(self.n_clusters)]
        self.vectors = vectors
        return self

    def search(self, query, k=10, nprobe=8, allowed=None):
        """nprobe = how many clusters to look inside. The recall/speed knob."""
        centroid_scores = self.centroids @ query
        nprobe = min(nprobe, self.n_clusters)
        if nprobe >= self.n_clusters:
            probe = np.arange(self.n_clusters)          # nprobe=all == exact search
        else:
            probe = np.argpartition(-centroid_scores, nprobe)[:nprobe]

        candidates = np.concatenate([self.lists[c] for c in probe if len(self.lists[c])])
        if allowed is not None:
            candidates = candidates[allowed[candidates]]      # pre-filter (see §6)
        if len(candidates) == 0:
            return np.array([], dtype=int)

        scores = self.vectors[candidates] @ query
        kk = min(k, len(candidates))
        top = np.argpartition(-scores, kk - 1)[:kk]
        return candidates[top[np.argsort(-scores[top])]]


print("Building IVF index (2048 clusters) - takes ~10-15 seconds...")
start = time.perf_counter()
index = IVFIndex(n_clusters=2048).build(vectors)
print("Index built in %.2f s" % (time.perf_counter() - start))
print()
sizes = [len(l) for l in index.lists]
print("Cluster sizes: min=%d  median=%d  max=%d" % (min(sizes), int(np.median(sizes)), max(sizes)))
print("Searching nprobe=8 of 2048 clusters touches roughly %.2f%% of the corpus."
      % (100 * 8 / 2048))

Building IVF index (2048 clusters) - takes ~10-15 seconds...
Index built in 9.29 s

Cluster sizes: min=1  median=253  max=4606
Searching nprobe=8 of 2048 clusters touches roughly 0.39% of the corpus.


---

## 6. The Recall/Speed Trade-off — The Number You Must Measure

Lesson 5 §10 insisted: *if you cannot state your recall, you do not know whether your
retrieval is broken.* So let's state it.

**Recall@10** = of the 10 truly-nearest vectors, how many did ANN return?

In [5]:
def recall_at_k(approx, exact, k=10):
    return len(set(approx[:k].tolist()) & set(exact[:k].tolist())) / k


print("%-8s %14s %12s %14s" % ("nprobe", "recall@10", "ms/query", "speedup vs exact"))
print("-" * 54)

results = []
for nprobe in [1, 2, 4, 8, 16, 32, 64, 128]:
    start = time.perf_counter()
    approx = [index.search(q, k=10, nprobe=nprobe) for q in queries]
    elapsed = time.perf_counter() - start

    ms = 1000 * elapsed / N_QUERIES
    rec = float(np.mean([recall_at_k(a, g) for a, g in zip(approx, ground_truth)]))
    results.append((nprobe, rec, ms))
    print("%-8d %14.3f %12.2f %14.1fx" % (nprobe, rec, ms, exact_ms / ms))

print()
good = [r for r in results if r[1] >= 0.95]
if good:
    nprobe, rec, ms = good[0]
    print(">>> Lowest nprobe reaching the 0.95 recall target: nprobe=%d" % nprobe)
    print(">>> recall=%.3f at %.2f ms/query - a %.0fx speedup over exact search,"
          % (rec, ms, exact_ms / ms))
    print(">>> for %.1f%% of the recall." % (100 * rec))

nprobe        recall@10     ms/query speedup vs exact
------------------------------------------------------
1                 0.422         0.40           48.6x
2                 0.672         0.66           29.5x
4                 0.874         0.92           21.3x
8                 0.990         1.13           17.3x
16                1.000         1.76           11.1x
32                1.000         4.42            4.4x
64                1.000         7.53            2.6x
128               1.000        21.17            0.9x

>>> Lowest nprobe reaching the 0.95 recall target: nprobe=8
>>> recall=0.990 at 1.13 ms/query - a 17x speedup over exact search,
>>> for 99.0% of the recall.


### Reading This Table — The Q5 Answer

Three things are visible in the numbers above, and they're the whole argument for ANN:

1. **Recall climbs steeply, then flattens.** The first few `nprobe` increments buy a lot of
   recall. Beyond the knee you pay linearly more time for almost nothing.
2. **Speedup decays toward 1×.** By `nprobe = 128` the index is examining so many
   clusters that it has no advantage left over a single optimised matrix multiply. At
   `nprobe = 2048` it would search every cluster — that *is* exact search with extra
   overhead. Seeing recall reach exactly 1.000 confirms the implementation is correct.
3. **The knee is where you operate.** You tune to the smallest setting that clears your
   recall target, and you re-measure whenever the data distribution changes.

`nprobe` here is exactly what `ef_search` is in HNSW: a **query-time** knob, so it can be
tuned without rebuilding the index. That is why it's the parameter you touch in production.

> ⚠️ Absolute timings depend on the machine — these ran single-threaded on a laptop CPU
> against 100k vectors. The **shape** of the curve is what transfers, not the milliseconds.

---

## 7. Q7 — The Post-Filter Failure, Demonstrated

The scenario: a cardiologist filters to `Department = Cardiology`, which is **2%** of the
corpus, and asks for 10 results.

Two implementations that sound equivalent:

- **Post-filter** — ANN retrieves top-N, *then* non-Cardiology results are discarded
- **Pre-filter** — the search is restricted to Cardiology vectors from the start

Let's run both.

In [6]:
is_cardiology = (departments == "Cardiology")
print("Cardiology documents: %s of %s (%.1f%%)"
      % ("{:,}".format(int(is_cardiology.sum())), "{:,}".format(N_VECTORS),
         100 * is_cardiology.mean()))
print()

K = 10
post_counts, pre_counts = [], []

for q in queries[:50]:
    # POST-FILTER: search normally, then throw away non-matching results
    raw = index.search(q, k=K, nprobe=8)
    post = [i for i in raw if is_cardiology[i]]
    post_counts.append(len(post))

    # PRE-FILTER: restrict candidates to Cardiology before scoring
    pre = index.search(q, k=K, nprobe=8, allowed=is_cardiology)
    pre_counts.append(len(pre))

print("Asked for %d results per query, across 50 queries:" % K)
print()
print("  POST-FILTER  average returned : %.2f  of %d" % (np.mean(post_counts), K))
print("               queries returning ZERO results : %d / 50"
      % sum(1 for c in post_counts if c == 0))
print("               queries returning fewer than %d : %d / 50"
      % (K, sum(1 for c in post_counts if c < K)))
print()
print("  PRE-FILTER   average returned : %.2f  of %d" % (np.mean(pre_counts), K))
print("               queries returning ZERO results : %d / 50"
      % sum(1 for c in pre_counts if c == 0))
print("               queries returning fewer than %d : %d / 50"
      % (K, sum(1 for c in pre_counts if c < K)))

Cardiology documents: 19,846 of 1,000,000 (2.0%)

Asked for 10 results per query, across 50 queries:

  POST-FILTER  average returned : 0.22  of 10
               queries returning ZERO results : 40 / 50
               queries returning fewer than 10 : 50 / 50

  PRE-FILTER   average returned : 10.00  of 10
               queries returning ZERO results : 0 / 50
               queries returning fewer than 10 : 0 / 50


### Answer to Q7

The numbers above are the whole argument. **Post-filtering asked for 10 results and
returned almost none.**

The reason is simple arithmetic: if Cardiology is 2% of the corpus, then a top-10 result
set drawn from the *whole* corpus is expected to contain **0.2 Cardiology documents**.
Filtering afterwards leaves you with nothing — while relevant Cardiology documents sat
just outside the top 10, never examined.

**Why this is dangerous rather than merely wrong:** it fails *silently*. No error, no
exception. The API returns `200 OK` with a short list, the LLM receives little or no
context, and the doctor gets "I could not find information on that" for a question the
corpus answers perfectly well. It looks like a knowledge gap; it's an architecture bug.

**The fix**, and what to say in an interview:

> *"With a selective filter I need pre-filtering, because post-filtering silently returns
> fewer than k results. But naive pre-filtering can disconnect an HNSW graph, so I'd use a
> database that applies filters during graph traversal — Qdrant calls this filterable
> HNSW. I'd also add a regression test asserting we get k results back for a
> low-cardinality filter."*

The general rule: **the more selective the filter, the more post-filtering hurts.** At 50%
selectivity you may not notice. At 2% it is catastrophic.

---

## 8. Q8 & Q3 — Hybrid Search, Demonstrated

Now a small text corpus, to show a query where **pure vector search fails and hybrid
search succeeds**.

> ⚠️ **Honest disclosure:** we have no trained embedding model available offline, so
> embeddings are simulated with a hand-built concept dictionary — words map to medical
> concepts, and a document's vector is the average of its concepts. This reproduces the
> *behaviour* that matters (synonyms match; exact identifiers do not) but a real model
> learns this structure from data rather than being told. The BM25 half is a genuine,
> standard implementation.

In [7]:
import math
import re

# ---------------------------------------------------------------- corpus
DOCS = [
    ("DOC-001", "Management of acute myocardial infarction requires immediate reperfusion therapy", "Cardiology"),
    ("DOC-002", "Hypertension treatment guidelines recommend ACE inhibitors as first line", "Cardiology"),
    ("DOC-003", "PROTOCOL-CARD-2026-014 defines the cardiac catheterisation lab checklist", "Cardiology"),
    ("DOC-004", "Metoprolol dosing for adult patients with tachycardia and arrhythmia", "Cardiology"),
    ("DOC-005", "Chemotherapy regimens for advanced stage carcinoma of the lung", "Oncology"),
    ("DOC-006", "Radiation oncology planning for tumour targeting and margin definition", "Oncology"),
    ("DOC-007", "Paediatric dosing chart for analgesic administration in children", "Pediatrics"),
    ("DOC-008", "Neonatal jaundice assessment and phototherapy thresholds", "Pediatrics"),
    ("DOC-009", "Renal insufficiency staging and dialysis referral criteria", "Nephrology"),
    ("DOC-010", "Metformin contraindications in patients with impaired kidney function", "Pharmacy"),
    ("DOC-011", "Contrast agent protocols for computed tomography imaging", "Radiology"),
    ("DOC-012", "Billing code submission workflow for outpatient procedures", "Billing"),
]

# ------------------------------------------------- simulated embedding model
CONCEPTS = ["cardiac", "cancer", "child", "kidney", "drug", "imaging", "admin", "treatment"]
CONCEPT_IDX = {c: i for i, c in enumerate(CONCEPTS)}

# The synonym knowledge a trained embedding model would have learned from data
WORD_CONCEPTS = {
    "myocardial": "cardiac", "infarction": "cardiac", "cardiac": "cardiac",
    "heart": "cardiac", "attack": "cardiac", "hypertension": "cardiac",
    "tachycardia": "cardiac", "arrhythmia": "cardiac", "metoprolol": "cardiac",
    "catheterisation": "cardiac", "blood": "cardiac", "pressure": "cardiac",
    "carcinoma": "cancer", "chemotherapy": "cancer", "tumour": "cancer",
    "oncology": "cancer", "cancer": "cancer", "radiation": "cancer",
    "paediatric": "child", "children": "child", "neonatal": "child",
    "jaundice": "child", "child": "child", "infant": "child",
    "renal": "kidney", "kidney": "kidney", "dialysis": "kidney",
    "insufficiency": "kidney", "failure": "kidney", "nephrology": "kidney",
    "metformin": "drug", "analgesic": "drug", "dosing": "drug", "dose": "drug",
    "painkiller": "drug", "drug": "drug", "contraindications": "drug",
    "imaging": "imaging", "tomography": "imaging", "contrast": "imaging",
    "radiology": "imaging", "scan": "imaging",
    "billing": "admin", "code": "admin", "submission": "admin", "outpatient": "admin",
    "treatment": "treatment", "therapy": "treatment", "management": "treatment",
    "guidelines": "treatment", "protocol": "treatment", "regimens": "treatment",
    "reperfusion": "treatment", "phototherapy": "treatment",
}

def tokenize(text):
    return re.findall(r"[a-z0-9\-]+", text.lower())

def simulated_embed(text):
    """Average of the concept vectors for recognised words."""
    vec = np.zeros(len(CONCEPTS))
    for word in tokenize(text):
        concept = WORD_CONCEPTS.get(word)
        if concept:
            vec[CONCEPT_IDX[concept]] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

doc_vectors = np.array([simulated_embed(text) for _, text, _ in DOCS])

def vector_search(query, top_k=5):
    q = simulated_embed(query)
    if np.linalg.norm(q) == 0:
        return []                                  # no recognised concepts at all
    scores = doc_vectors @ q
    order = np.argsort(-scores)
    return [(DOCS[i][0], float(scores[i])) for i in order[:top_k] if scores[i] > 0]

print("Simulated embedding model ready: %d docs, %d concepts" % (len(DOCS), len(CONCEPTS)))

Simulated embedding model ready: 12 docs, 8 concepts


In [8]:
# ------------------------------------------------------------------ BM25
class BM25:
    """Standard Okapi BM25. This half is a genuine implementation."""

    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [tokenize(text) for _, text, _ in corpus]
        self.ids = [doc_id for doc_id, _, _ in corpus]
        self.N = len(self.docs)
        self.avgdl = sum(len(d) for d in self.docs) / self.N

        self.df = {}
        for doc in self.docs:
            for term in set(doc):
                self.df[term] = self.df.get(term, 0) + 1

    def idf(self, term):
        df = self.df.get(term, 0)
        return math.log(1 + (self.N - df + 0.5) / (df + 0.5))

    def search(self, query, top_k=5):
        q_terms = tokenize(query)
        scores = []
        for i, doc in enumerate(self.docs):
            score = 0.0
            for term in q_terms:
                f = doc.count(term)
                if f == 0:
                    continue
                denom = f + self.k1 * (1 - self.b + self.b * len(doc) / self.avgdl)
                score += self.idf(term) * f * (self.k1 + 1) / denom
            if score > 0:
                scores.append((self.ids[i], score))
        scores.sort(key=lambda x: -x[1])
        return scores[:top_k]


bm25 = BM25(DOCS)


def reciprocal_rank_fusion(rankings, k=60, top_k=5):
    """Merge rankings by POSITION, so incomparable score scales don't matter."""
    fused = {}
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking, start=1):
            fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k + rank)
    out = sorted(fused.items(), key=lambda x: -x[1])
    return out[:top_k]


def compare(query, note=""):
    print("QUERY: %r" % query)
    if note:
        print("       (%s)" % note)
    print("-" * 66)

    v = vector_search(query)
    b = bm25.search(query)
    h = reciprocal_rank_fusion([v, b])

    def show(label, results):
        if not results:
            print("  %-14s -> NOTHING FOUND" % label)
            return
        text = ", ".join("%s(%.3f)" % (d, s) for d, s in results[:3])
        print("  %-14s -> %s" % (label, text))

    show("VECTOR", v)
    show("BM25", b)
    show("HYBRID (RRF)", h)
    print()
    return v, b, h


print("=" * 66)
print("CASE 1 - Pure synonym: ZERO words in common with the target")
print("=" * 66)
compare("kidney failure", "target is DOC-009 'Renal insufficiency ... dialysis'")

print(">>> VECTOR gets it right: DOC-009 scores 1.000 even though it shares NO word")
print(">>> with the query - 'kidney failure' and 'renal insufficiency' are the same")
print(">>> concept. This is the synonym win that keyword search cannot reach.")
print(">>>")
print(">>> BM25 gets it WRONG: it matches the literal token 'kidney' in DOC-010,")
print(">>> a metformin document, and never sees DOC-009 at all.")

CASE 1 - Pure synonym: ZERO words in common with the target
QUERY: 'kidney failure'
       (target is DOC-009 'Renal insufficiency ... dialysis')
------------------------------------------------------------------
  VECTOR         -> DOC-009(1.000), DOC-010(0.447)
  BM25           -> DOC-010(2.149)
  HYBRID (RRF)   -> DOC-010(0.033), DOC-009(0.016)

>>> VECTOR gets it right: DOC-009 scores 1.000 even though it shares NO word
>>> with the query - 'kidney failure' and 'renal insufficiency' are the same
>>> concept. This is the synonym win that keyword search cannot reach.
>>>
>>> BM25 gets it WRONG: it matches the literal token 'kidney' in DOC-010,
>>> a metformin document, and never sees DOC-009 at all.


In [9]:
print("=" * 66)
print("CASE 2 - Exact protocol identifier  <<< THE Q8 ANSWER")
print("=" * 66)
v, b, h = compare("PROTOCOL-CARD-2026-014", "target is DOC-003")

print(">>> VECTOR SEARCH FAILS. The identifier contains no medical concept words,")
print(">>> so the query embeds to a near-meaningless vector. Even a real model would")
print(">>> place 'PROTOCOL-CARD-2026-014' and 'PROTOCOL-CARD-2026-015' almost on top")
print(">>> of each other - the gist of two protocol numbers is identical.")
print(">>> BM25 matches the exact token instantly, and RRF keeps it at the top.")
print()

print("=" * 66)
print("CASE 3 - Drug name precision: a patient-safety case")
print("=" * 66)
compare("metformin kidney contraindications", "metformin (DOC-010), NOT metoprolol (DOC-004)")

print("=" * 66)
print("CASE 4 - Conceptual query with no exact keyword overlap")
print("=" * 66)
compare("painkiller dose for children", "target is DOC-007 'analgesic ... children'")

CASE 2 - Exact protocol identifier  <<< THE Q8 ANSWER
QUERY: 'PROTOCOL-CARD-2026-014'
       (target is DOC-003)
------------------------------------------------------------------
  VECTOR         -> NOTHING FOUND
  BM25           -> DOC-003(2.278)
  HYBRID (RRF)   -> DOC-003(0.016)

>>> VECTOR SEARCH FAILS. The identifier contains no medical concept words,
>>> so the query embeds to a near-meaningless vector. Even a real model would
>>> place 'PROTOCOL-CARD-2026-014' and 'PROTOCOL-CARD-2026-015' almost on top
>>> of each other - the gist of two protocol numbers is identical.
>>> BM25 matches the exact token instantly, and RRF keeps it at the top.

CASE 3 - Drug name precision: a patient-safety case
QUERY: 'metformin kidney contraindications'
       (metformin (DOC-010), NOT metoprolol (DOC-004))
------------------------------------------------------------------
  VECTOR         -> DOC-010(1.000), DOC-007(0.632), DOC-009(0.447)
  BM25           -> DOC-010(6.448)
  HYBRID (RRF)   -> DOC

### ⚠️ What Case 1 Also Reveals About RRF — An Honest Reading

Look carefully at Case 1's hybrid row: **RRF ranks DOC-010 above DOC-009**, even though
DOC-009 is the correct answer and vector search ranked it first with a perfect score.

That is not a bug in the code — it is RRF working exactly as designed, and it is worth
understanding rather than hiding:

- DOC-009 was found by **one** retriever (vector, rank 1) → `1/61 = 0.0164`
- DOC-010 was found by **both** (vector rank 2, BM25 rank 1) → `1/62 + 1/61 = 0.0325`

**RRF rewards consensus.** A document both retrievers like outranks a document only one
retriever likes, even when that one retriever was confident and correct.

Two honest conclusions:

1. **This effect is exaggerated here** because the corpus has 12 documents and only two
   retrievers. At realistic scale, with top-50 lists from each retriever, a genuinely
   correct document usually surfaces in both.
2. **This is precisely why production RAG puts a reranker after fusion.** RRF is a cheap,
   scale-free way to merge candidate lists — it is *not* a relevance judge. The
   cross-encoder reranker in the §1 architecture reads the query and each candidate
   *together* and makes the final call. Fusion widens the net; the reranker decides.

If you took away only "hybrid search is strictly better", this case is the correction.
Hybrid search improves **recall** — the chance the right document is somewhere in your
candidate set. Turning that into the right **ranking** is the reranker's job.

### Answer to Q8

**A query where hybrid wins and pure vector search fails:** `PROTOCOL-CARD-2026-014`
(Case 2 above).

The identifier carries no semantic content. Vector search returns nothing useful, because
embeddings encode *meaning* and one protocol number means essentially the same thing as
any other. BM25 treats it as an exact token and finds it immediately. RRF fuses the two
and keeps the correct document at rank 1.

**The general principle, and the Q3 answer:**

| Query type | Vector | BM25 | Why |
| --- | --- | --- | --- |
| `heart attack treatment` | ✅ | ❌ | Synonym — no shared words with "myocardial infarction" |
| `PROTOCOL-CARD-2026-014` | ❌ | ✅ | Exact token — no semantic content to embed |
| `metformin contraindications` | ⚠️ | ✅ | Drug names are near-identical in vector space |
| `painkiller dose for children` | ✅ | ❌ | "painkiller" ≈ "analgesic", zero overlap |

Neither retriever wins everywhere, and real doctors type all four kinds of query. **That
is the argument for hybrid search** — not that it's more sophisticated, but that either
method alone has a category of query it reliably fails.

In a hospital the stakes sharpen it: confusing **metoprolol** (a beta blocker) with
**metformin** (a diabetes drug) is a patient-safety incident, not a relevance miss.

---

## 9. Summary

### The Eight Answers

| # | Question | Answer |
| --- | --- | --- |
| 1 | Why a vector database? | Doctors use natural language; clinical synonyms share no characters. Plus ANN indexing avoids scanning 40M vectors per query |
| 2 | Which retrieval method? | HNSW, `M=16`, `ef_construction=200`, `ef_search` tuned to ≥0.95 measured recall, filterable for metadata |
| 3 | Hybrid search? | **Yes, mandatory** — drug names, ICD-10 codes, and protocol IDs need exact matching |
| 4 | Metadata filtering benefit? | Speed (50× smaller space), accuracy (no paediatric doses for adults), and **access control** — the real reason |
| 5 | Why ANN over exact? | Measured in §6: large speedup for a few percent of recall |
| 6 | RAM for 5M docs? | ~**40M chunks**, ≈200 GB at 1024-dim float32. Shard by Country + scalar quantisation |
| 7 | Post-filter at 2%? | Demonstrated in §7 — returns almost nothing, and fails **silently** |
| 8 | Hybrid-wins query? | `PROTOCOL-CARD-2026-014` — no semantic content to embed |

### What I'd Do Before Calling This Production-Ready

| Gap | Why it matters |
| --- | --- |
| **Medical golden set** (100+ doctor questions → correct documents) | The only way to know retrieval actually works. Nothing else substitutes |
| **Recall regression test in CI** | Index recall silently degrades as data grows |
| **Superseded-protocol handling** | An outdated clinical protocol is worse than no answer |
| **PHI detection at ingestion** | Vectors are personal data — inversion attacks are real (Lesson 4) |
| **Audit logging of every retrieval** | Required under HIPAA; also the only way to debug a complaint |
| **Human-in-the-loop for clinical answers** | This should assist a doctor's judgement, never replace it |
| **Country sharding** | Data-residency law, plus it solves the 200 GB problem |

### The Lesson

Three things in this notebook cannot be learned from a diagram, and all three fail
**silently** in production:

1. **Post-filtering at low selectivity returns nothing** — and returns HTTP 200 while
   doing it (§7).
2. **Recall is a number you measure, not a property you assume** (§6).
3. **Pure vector search cannot find an identifier**, no matter how good the model (§8).

Each looks like "the AI didn't know the answer". None of them is.

---

**Next lesson:** *RAG from Beginner to Production* — the complete pipeline, chunking
strategies, reranking, evaluation, and the failures that only appear at scale.